# Transformadas Geométricas con OpenCV

En este notebook aplicamos las principales **transformadas geométricas afines** a una imagen de medusas utilizando OpenCV.

Las transformadas afines se representan mediante una matriz de $2 \times 3$ que se aplica con `cv2.warpAffine`. La forma general es:

$$
\begin{bmatrix} x' \\ y' \end{bmatrix} = \begin{bmatrix} a_{11} & a_{12} & t_x \\ a_{21} & a_{22} & t_y \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

**Transformadas cubiertas:** Identidad, Escala, Reflexión, Rotación, Translación, Shear Horizontal y Shear Vertical.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Configuracion ---
# Directorio donde vive este notebook y donde se guardan los resultados
DIR_BASE = os.path.dirname(os.path.abspath("__file__"))
RUTA_IMAGEN = os.path.join(DIR_BASE, "img1.jpg")

# Cargar imagen en BGR (formato nativo de OpenCV)
img_bgr = cv2.imread(RUTA_IMAGEN)
assert img_bgr is not None, f"No se pudo cargar la imagen en: {RUTA_IMAGEN}"

# Convertir a RGB para visualizar correctamente con matplotlib
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

alto, ancho = img_bgr.shape[:2]
print(f"Imagen cargada: {ancho} x {alto} pixeles")

# Mostrar imagen original
plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.title("Imagen Original")
plt.axis("off")
plt.tight_layout()
plt.show()

## 1. Transformada Identidad

La matriz identidad no modifica la imagen. Sirve como caso base para verificar que el pipeline funciona correctamente.

$$
M_{\text{identidad}} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}
$$

Cada pixel $(x, y)$ se mapea a si mismo: $(x', y') = (x, y)$.

In [ ]:
# Matriz identidad 2x3
M_identidad = np.float32([[1, 0, 0],
                          [0, 1, 0]])

t1 = cv2.warpAffine(img_bgr, M_identidad, (ancho, alto),
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

# Guardar resultado
cv2.imwrite(os.path.join(DIR_BASE, "t1_identidad.jpg"), t1)

# Mostrar
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(t1, cv2.COLOR_BGR2RGB))
plt.title("T1 - Identidad")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Matriz aplicada:")
print(M_identidad)

## 2. Escala y Reflexión

### 2a. Escala

Reducimos la imagen al **60%** en ambos ejes ($s_x = 0.6$, $s_y = 0.6$):

$$
M_{\text{escala}} = \begin{bmatrix} 0.6 & 0 & 0 \\ 0 & 0.6 & 0 \end{bmatrix}
$$

### 2b. Reflexión Horizontal

Invertimos el eje $x$ multiplicando por $-1$ y trasladando para que la imagen permanezca visible:

$$
M_{\text{reflexion}} = \begin{bmatrix} -1 & 0 & w-1 \\ 0 & 1 & 0 \end{bmatrix}
$$

donde $w$ es el ancho de la imagen. El desplazamiento $w-1$ compensa la inversion para que la imagen no quede fuera del lienzo.

In [ ]:
# --- Escala al 60% ---
sx, sy = 0.6, 0.6
M_escala = np.float32([[sx, 0, 0],
                       [0, sy, 0]])

t2_escala = cv2.warpAffine(img_bgr, M_escala, (ancho, alto),
                           borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t2_escala.jpg"), t2_escala)

# --- Reflexion horizontal ---
M_reflexion = np.float32([[-1, 0, ancho - 1],
                          [ 0, 1, 0]])

t2_reflexion = cv2.warpAffine(img_bgr, M_reflexion, (ancho, alto),
                              borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t2_reflexion.jpg"), t2_reflexion)

# Mostrar ambas
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(cv2.cvtColor(t2_escala, cv2.COLOR_BGR2RGB))
axes[0].set_title("T2a - Escala (60%)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(t2_reflexion, cv2.COLOR_BGR2RGB))
axes[1].set_title("T2b - Reflexion Horizontal")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Matriz de escala:")
print(M_escala)
print("\nMatriz de reflexion:")
print(M_reflexion)

## 3. Rotación

Rotamos la imagen **45 grados** en sentido antihorario alrededor de su centro.

La matriz de rotacion general con centro $(c_x, c_y)$, angulo $\theta$ y escala $s$ es:

$$
M_{\text{rotacion}} = \begin{bmatrix} s \cos\theta & s \sin\theta & (1 - s\cos\theta) \cdot c_x - s \sin\theta \cdot c_y \\ -s \sin\theta & s \cos\theta & s \sin\theta \cdot c_x + (1 - s\cos\theta) \cdot c_y \end{bmatrix}
$$

Usamos `cv2.getRotationMatrix2D(center, angle=45, scale=1.0)` que construye esta matriz automaticamente.

In [ ]:
# Centro de la imagen
centro = (ancho / 2, alto / 2)

# Obtener matriz de rotacion: 45 grados, sin cambio de escala
M_rotacion = cv2.getRotationMatrix2D(centro, angle=45, scale=1.0)

t3 = cv2.warpAffine(img_bgr, M_rotacion, (ancho, alto),
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t3_rotacion.jpg"), t3)

# Mostrar
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(t3, cv2.COLOR_BGR2RGB))
plt.title("T3 - Rotacion 45 grados")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Matriz de rotacion:")
print(M_rotacion)

## 4. Translación

Desplazamos la imagen $t_x = 100$ pixeles a la derecha y $t_y = 80$ pixeles hacia abajo.

$$
M_{\text{translacion}} = \begin{bmatrix} 1 & 0 & 100 \\ 0 & 1 & 80 \end{bmatrix}
$$

Los pixeles que salen del lienzo se pierden; los que no tienen origen se rellenan con negro.

In [ ]:
# Translacion: 100 px derecha, 80 px abajo
tx, ty = 100, 80
M_translacion = np.float32([[1, 0, tx],
                            [0, 1, ty]])

t4 = cv2.warpAffine(img_bgr, M_translacion, (ancho, alto),
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t4_translacion.jpg"), t4)

# Mostrar
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(t4, cv2.COLOR_BGR2RGB))
plt.title("T4 - Translacion (tx=100, ty=80)")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Matriz de translacion:")
print(M_translacion)

## 5. Shear Horizontal

El shear (cizallamiento) horizontal desplaza cada fila de pixeles proporcionalmente a su posicion vertical, con factor $sh_x = 0.3$.

$$
M_{\text{shear\_h}} = \begin{bmatrix} 1 & 0.3 & 0 \\ 0 & 1 & 0 \end{bmatrix}
$$

El efecto visual es una inclinacion de la imagen hacia la derecha.

In [ ]:
# Shear horizontal con factor 0.3
shx = 0.3
M_shear_h = np.float32([[1, shx, 0],
                        [0,   1, 0]])

t5 = cv2.warpAffine(img_bgr, M_shear_h, (ancho, alto),
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t5_shear_h.jpg"), t5)

# Mostrar
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(t5, cv2.COLOR_BGR2RGB))
plt.title("T5 - Shear Horizontal (shx=0.3)")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Matriz de shear horizontal:")
print(M_shear_h)

## 6. Shear Vertical

El shear vertical desplaza cada columna de pixeles proporcionalmente a su posicion horizontal, con factor $sh_y = 0.3$.

$$
M_{\text{shear\_v}} = \begin{bmatrix} 1 & 0 & 0 \\ 0.3 & 1 & 0 \end{bmatrix}
$$

El efecto visual es una inclinacion de la imagen hacia abajo.

In [ ]:
# Shear vertical con factor 0.3
shy = 0.3
M_shear_v = np.float32([[1,   0, 0],
                        [shy, 1, 0]])

t6 = cv2.warpAffine(img_bgr, M_shear_v, (ancho, alto),
                     borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))

cv2.imwrite(os.path.join(DIR_BASE, "t6_shear_v.jpg"), t6)

# Mostrar
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(t6, cv2.COLOR_BGR2RGB))
plt.title("T6 - Shear Vertical (shy=0.3)")
plt.axis("off")
plt.tight_layout()
plt.show()

print("Matriz de shear vertical:")
print(M_shear_v)

## Comparacion Visual

A continuacion se muestran todas las transformadas lado a lado para facilitar la comparacion.

In [ ]:
# Cuadricula comparativa de todas las transformadas
resultados = [
    ("Original",              img_bgr),
    ("T1 - Identidad",       t1),
    ("T2a - Escala (60%)",   t2_escala),
    ("T2b - Reflexion H.",   t2_reflexion),
    ("T3 - Rotacion 45\u00b0",    t3),
    ("T4 - Translacion",     t4),
    ("T5 - Shear H.",       t5),
    ("T6 - Shear V.",       t6),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, (titulo, imagen) in zip(axes.flatten(), resultados):
    ax.imshow(cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB))
    ax.set_title(titulo, fontsize=12, fontweight="bold")
    ax.axis("off")

plt.suptitle("Transformadas Geometricas Aplicadas", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("Todas las imagenes fueron guardadas en:")
print(DIR_BASE)